In [ ]:
pip install openai

In [ ]:
import time,json
from cubestudio.request.model_client import Client,init
from cubestudio.train.task import InferenceService,Project

In [ ]:
# 初始化认证
import os
HOST = os.environ['MODELONE_API_URL']
token = os.environ['MODELONE_API_TOKEN']
username = os.environ.get('MODELONE_USERNAME', 'admin')
init(host=HOST,username=username,token=token)


In [ ]:
# 添加一个inferenceservice
inferenceservice = Client(InferenceService).add_or_update(
    service_type=f'vllm',
    project=Client(Project).one(name='public'),
    label='chatglm3-6b对话模型',
    model_name='chatglm3-6b',
    model_version='v202300801',
    model_path='/mnt/admin/pipeline/example/sdk/chatglm/chatglm3-6b',
    images='vllm/vllm-openai:v0.8.5.post1',
    resource_memory='20G',
    host="/v1/models",
    resource_cpu='10',
    resource_gpu='1',
    min_replicas='1',
    max_replicas='1',
    ports='8000',
    volume_mount='kubeflow-user-workspace(pvc):/mnt',
    working_dir='',
    command='python -m vllm.entrypoints.openai.api_server --trust-remote-code --model $KUBEFLOW_MODEL_PATH --host 0.0.0.0 --port 8000 --dtype float16 --tensor-parallel-size $RESOURCE_GPU --served-model-name chatglm3-6b',
    env='HF_ENDPOINT=https://hf-mirror.com',
)

In [ ]:
print(json.dumps(inferenceservice.to_dict(), indent=4))
if inferenceservice.model_status!='online':
    inferenceservice.deploy()

In [ ]:
import requests
import json

url = "http://chatglm3-6b-202300801.service:8000/v1/chat/completions"

payload = json.dumps({
  "model": "chatglm3-6b",
  "messages": [
    {
      "role": "user",
      "content": "介绍一下北京。"
    }
  ],
  "temperature": 0,
  "stop": "<|im_end|>"
})

response = requests.request("POST", url, data=payload)
content = response.json()['choices'][0]['message']['content']
print(content)


In [ ]:
inferenceservice.clear()